In [0]:
from pyspark.sql import Row

# Sample data for pivot
data = [
    Row(category="A", year=2024, value=10),
    Row(category="A", year=2025, value=15),
    Row(category="B", year=2024, value=20),
    Row(category="B", year=2025, value=25),
    Row(category="C", year=2024, value=30),
    Row(category="C", year=2025, value=35)
]

df = spark.createDataFrame(data)

# View of current data
display(df)

# Pivot data: years as columns, values as cell values
pivoted_df = df.groupBy("category").pivot("year").sum("value")

# View of expected data after pivot
display(pivoted_df)

In [0]:
from pyspark.sql import Row

# Sample data for pivot
data = [
    Row(category="A", year_=2024, values=10),
    Row(category="A", year_=2025, values=15),
    Row(category="B", year_=2024, values=20),
    Row(category="B", year_=2025, values=25),
    Row(category="C", year_=2024, values=30),
    Row(category="C", year_=2025, values=35)
]

df = spark.createDataFrame(data)
df.createOrReplaceTempView('tmp_tbl')
display(df)

In [0]:
spark.sql("""  select *  from tmp_tbl""").show(truncate=False)

In [0]:
spark.sql("""  select category, case when year_='2024' then values else null end as y_2024,
          case when year_='2025' then values else null end as y_2025 from tmp_tbl """).show(truncate=False)

In [0]:
spark.sql("""  select category, sum(case when year_='2024' then values else null end) as y_2024,
          sum(case when year_='2025' then values else null end) as y_2025 from tmp_tbl group by category """).show(truncate=False)

In [0]:
spark.sql("""  select * from tmp_tbl """).show(truncate=False)

In [0]:
spark.sql("""  select year_, case when category='A' then values else null end as cat_A,
          case when category='B' then values else null end as cat_B,
          case when category='C' then values else null end as cat_C
           from tmp_tbl """).show(truncate=False)

In [0]:
spark.sql("""  select year_, sum(case when category='A' then values else null end) as cat_A,
          sum(case when category='B' then values else null end) as cat_B,
          sum(case when category='C' then values else null end) as cat_C
           from tmp_tbl group by year_ """).show(truncate=False)

In [0]:
from pyspark.sql import Row
from datetime import datetime, timedelta

# Sample customers data
customers_data = [
    Row(Customer_Id=1, First_Name="Alice", Last_Name="Smith", Email="alice@example.com"),
    Row(Customer_Id=2, First_Name="Bob", Last_Name="Jones", Email="bob@example.com"),
    Row(Customer_Id=3, First_Name="Charlie", Last_Name="Brown", Email="charlie@example.com"),
    Row(Customer_Id=4, First_Name="Diana", Last_Name="Prince", Email="diana@example.com")
]

customers_df = spark.createDataFrame(customers_data)
customers_df.createOrReplaceTempView('customers')
display(customers_df)

# Sample order details data
today = datetime(2026, 7, 21)
orders_data = [
    Row(Order_Id=101, Customer_Id=1, Order_Date=(today - timedelta(days=10)).strftime('%Y-%m-%d')),
    Row(Order_Id=102, Customer_Id=2, Order_Date=(today - timedelta(days=95)).strftime('%Y-%m-%d')),
    Row(Order_Id=103, Customer_Id=1, Order_Date=(today - timedelta(days=50)).strftime('%Y-%m-%d')),
    Row(Order_Id=104, Customer_Id=3, Order_Date=(today - timedelta(days=120)).strftime('%Y-%m-%d'))
]

orders_df = spark.createDataFrame(orders_data)
orders_df.createOrReplaceTempView('orders')
display(orders_df)


In [0]:
spark.sql("""select customer_id,first_name,last_name,current_date() as cnt_dt,date_sub(current_date(),100) as thirty_days_ago from customers 
          where customer_id  not in (
              select distinct customer_id from orders where order_date between date_sub(current_date(),100) and current_date())
              
          """).show(truncate=False)

In [0]:
spark.sql(""" select c.customer_id,c.first_name,c.last_name,o.order_date from customers c left join orders o on c.customer_id=o.customer_id  where o.order_date not between date_sub(current_date(),100) and current_date() """).show(truncate=False)

In [0]:
spark.sql("""
SELECT department_id, MAX(salary) AS second_highest_salary
FROM my_learning.my_raw_feed.employees
WHERE salary < (
    SELECT MAX(salary)
    FROM employees e2
    WHERE e2.department_id = employees.department_id
)
GROUP BY department_id
""").show(truncate=False)